# 🏨 LodgeTrust Aesthetic Model Fine-Tuning (Google Colab)

This notebook fine-tunes the **`cafeai/cafe_aesthetic`** vision transformer on your custom 500 room image aesthetic dataset using Google Colab's free GPU (T4/V100).

---

## 1️⃣ Verify GPU and Mount Google Drive

In [ ]:
!nvidia-smi

from google.colab import drive
drive.mount('/content/drive')

## 2️⃣ Install Required Dependencies

In [ ]:
!pip install -q transformers datasets accelerate pillow torchvision evaluate

## 3️⃣ Locate Workspace in Google Drive

Set `WORKSPACE_DIR` to where you uploaded `fine_tune_aesthetic` in your Drive.
Default: `/content/drive/MyDrive/fine_tune_aesthetic`

In [ ]:
import os
from pathlib import Path

# Set workspace path (adjust if your folder in Drive has a different name)
WORKSPACE_DIR = Path('/content/drive/MyDrive/fine_tune_aesthetic')

# Verify files exist
data_dir = WORKSPACE_DIR / 'data'
train_csv = data_dir / 'train_labels.csv'
val_csv = data_dir / 'val_labels.csv'
images_dir = data_dir / 'images'

print(f"Workspace directory: {WORKSPACE_DIR}")
print(f"Train CSV exists: {train_csv.exists()}")
print(f"Val CSV exists: {val_csv.exists()}")
print(f"Images directory exists: {images_dir.exists()} ({len(list(images_dir.glob('*.jpg')))} images found)")

## 4️⃣ Run Fine-Tuning Script with GPU Acceleration

In [ ]:
%cd /content/drive/MyDrive/fine_tune_aesthetic
!python scripts/fine_tune_aesthetic.py --epochs 5 --batch_size 16 --lr 5e-5

## 5️⃣ Test Inference with the Fine-Tuned Model

In [ ]:
import torch
from PIL import Image
from transformers import AutoImageProcessor, AutoModelForImageClassification

MODEL_DIR = WORKSPACE_DIR / 'model'
print(f"Loading fine-tuned model from: {MODEL_DIR}")

device = 'cuda' if torch.cuda.is_available() else 'cpu'
processor = AutoImageProcessor.from_pretrained(str(MODEL_DIR))
model = AutoModelForImageClassification.from_pretrained(str(MODEL_DIR)).to(device)
model.eval()

# Test on a sample room image
sample_img_path = list(images_dir.glob('*.jpg'))[0]
img = Image.open(sample_img_path).convert('RGB')

inputs = processor(images=img, return_tensors='pt').to(device)
with torch.no_grad():
    outputs = model(**inputs)
    score = outputs.logits.squeeze().cpu().item()
    score = round(max(1.0, min(10.0, float(score))), 2)

print(f"\n🖼️ Sample image: {sample_img_path.name}")
print(f"⭐ Predicted Aesthetic Score: {score} / 10.0")
display(img.resize((350, 250)))

## 🎉 Next Steps
The fine-tuned model weights (`model.safetensors`, `config.json`, `preprocessor_config.json`) are now saved directly in your Google Drive at:
**`fine_tune_aesthetic/model/`**

When you download the `model/` folder back to your local `project/fine_tune_aesthetic/model/`, the LodgeTrust backend and frontend will automatically use your fine-tuned model checkpoint!